# Nassau Candy — Lead-Time Model Training

Rebuilds the lead-time model used in the Streamlit app and reports honest test-set metrics.

**Data:** `cleaned_candy_data.csv` (2,051 order lines). Rows with no distance (58 Canadian orders) are dropped.

**Split:** 80% train, 20% test, `random_state=42`.

In [1]:
import pandas as pd, numpy as np, joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('cleaned_candy_data.csv')
print(df.shape)
print('Missing distance:', df['Distance_km'].isna().sum())
df = df.dropna(subset=['Distance_km'])

(2051, 25)
Missing distance: 58


## Data check
Lead time = Ship Date − Order Date. Orders are from 2024 but ship dates are in 2026–27, so lead times sit around 908 days. The **differences** between orders (a few days) are what the model learns.

In [2]:
print(df['Lead Time'].describe())
print('Correlation, distance vs lead time:', round(df['Distance_km'].corr(df['Lead Time']), 3))
print(df.groupby('Ship Mode')['Lead Time'].mean().round(1))

count    1993.000000
mean      908.059709
std         1.682639
min       904.000000
25%       907.000000
50%       908.000000
75%       909.000000
max       912.000000
Name: Lead Time, dtype: float64
Correlation, distance vs lead time: 0.015
Ship Mode
First Class       906.3
Same Day          904.0
Second Class      907.3
Standard Class    909.0
Name: Lead Time, dtype: float64


## Features
Same 14 columns the app's model expects: distance, sales, units, and one-hot columns for division, region and ship mode.

In [3]:
X = pd.get_dummies(df[['Distance_km','Sales','Units','Division','Region','Ship Mode']],
                   columns=['Division','Region','Ship Mode']).astype(float)
y = df['Lead Time']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
list(X.columns)

['Distance_km',
 'Sales',
 'Units',
 'Division_Chocolate',
 'Division_Other',
 'Division_Sugar',
 'Region_Atlantic',
 'Region_Gulf',
 'Region_Interior',
 'Region_Pacific',
 'Ship Mode_First Class',
 'Ship Mode_Same Day',
 'Ship Mode_Second Class',
 'Ship Mode_Standard Class']

## Compare models against two simple baselines
A model is only useful if it beats a simple rule. Baseline 1 predicts the overall average. Baseline 2 predicts the average lead time for each ship mode.

In [4]:
def score(name, pred):
    return {'Model': name,
            'MAE (days)': round(mean_absolute_error(y_test, pred), 2),
            'RMSE (days)': round(mean_squared_error(y_test, pred) ** 0.5, 2),
            'R2': round(r2_score(y_test, pred), 2)}

rows = [score('Baseline: overall average', np.full(len(y_test), y_train.mean()))]
mode_avg = df.loc[X_train.index].groupby('Ship Mode')['Lead Time'].mean()
rows.append(score('Baseline: average per ship mode', df.loc[X_test.index, 'Ship Mode'].map(mode_avg)))

models = {'Linear Regression': LinearRegression(),
          'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
          'Gradient Boosting': GradientBoostingRegressor(random_state=42)}
for name, m in models.items():
    m.fit(X_train, y_train)
    rows.append(score(name, m.predict(X_test)))
pd.DataFrame(rows)

,Model,MAE (days),RMSE (days),R2
0,Baseline: overall average,1.33,1.74,-0.00
1,Baseline: average per ship mode,0.82,1.02,0.66
2,Linear Regression,0.84,1.02,0.66
3,Random Forest,0.87,1.11,0.59
4,Gradient Boosting,0.85,1.03,0.65


## What drives the Random Forest?

In [5]:
rf = models['Random Forest']
pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(6).round(3)

Ship Mode_Standard Class    0.580
Ship Mode_Same Day          0.120
Distance_km                 0.116
Sales                       0.092
Units                       0.030
Ship Mode_Second Class      0.016
dtype: float64

## Save the model for the app

In [6]:
joblib.dump(rf, 'candy_model.pkl')
joblib.dump(list(X.columns), 'model_columns.pkl')

['model_columns.pkl']

## Findings
- Ship mode explains most of the variation in lead time.
- Distance has almost no link to lead time (correlation ≈ 0.015).
- Random Forest does **not** beat the simple per-ship-mode average. Linear Regression and Gradient Boosting are level with it.